## Developmental regimes under asymmetric training conditions for CIFAR-10

This notebook can be used to replicate the results reported in  "Generative adversarial learning can explain why imagination seems less real as we grow up" by Ozsu, Petrova, Dekker & Dijkstra.

This script would run the same training procedure as reported in the main text yet, it will use models from later stages of training create an asymmetry in training conditions to investigate if co-evolution from untrained baseline is necessary for gathering developmental regimes that could explain behaviour. The results of this experiment is reported in Figure 7.

To run this scripts, you would need to have the weights available which can be gathered from the OSF files. In this implementation, the weights are imported via Google Drive but other methods are also possible.

For any questions or if you detect a bug: a.ozsu@ucl.ac.uk


---



In [ ]:
!nvidia-smi

Thu Mar 12 11:37:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip "/content/drive/MyDrive/checkpoints_newarch.zip" -d "/content"

Mounted at /content/drive
Archive:  /content/drive/MyDrive/checkpoints_newarch.zip
   creating: /content/checkpoints/
  inflating: /content/checkpoints/C_epoch_1.pth  
  inflating: /content/checkpoints/C_epoch_10.pth  
  inflating: /content/checkpoints/C_epoch_100.pth  
  inflating: /content/checkpoints/C_epoch_11.pth  
  inflating: /content/checkpoints/C_epoch_12.pth  
  inflating: /content/checkpoints/C_epoch_13.pth  
  inflating: /content/checkpoints/C_epoch_14.pth  
  inflating: /content/checkpoints/C_epoch_15.pth  
  inflating: /content/checkpoints/C_epoch_16.pth  
  inflating: /content/checkpoints/C_epoch_17.pth  
  inflating: /content/checkpoints/C_epoch_18.pth  
  inflating: /content/checkpoints/C_epoch_19.pth  
  inflating: /content/checkpoints/C_epoch_2.pth  
  inflating: /content/checkpoints/C_epoch_20.pth  
  inflating: /content/checkpoints/C_epoch_21.pth  
  inflating: /content/checkpoints/C_epoch_22.pth  
  inflating: /content/checkpoints/C_epoch_23.pth  
  inflating: /co

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
import os
import pickle
import cv2
import warnings
from timeit import default_timer as timer

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """
    Central configuration for all hyperparameters.
    This class holds all the tunable parameters for the GAN, making it easy to
    see and modify the experimental setup from one place.
    """

    # --- 1. Dataset and Data Paths ---
    dataset_name = "cifar10"  # Options: 'cifar10', 'mnist'
    data_path = "./data"

    # --- 2. Model Architecture ---
    latent_dim = 128
    generator_feature_maps = 128
    critic_feature_maps = 128
    image_channels = 3

    # --- 3. WGAN-GP Training Parameters ---
    batch_size = 64
    num_epochs = 90
    learning_rate_generator = 0.0001
    learning_rate_critic = 0.0003
    adam_beta1 = 0.0
    adam_beta2 = 0.9
    critic_updates_per_generator_update = 1

    # --- 4. Stability and Regularization ---
    gradient_penalty_weight = 10


    # --- 5. Monitoring, Logging, and Saving ---
    monitor_every_steps = 50
    save_samples_every_steps = 200
    save_model_every_epochs = 5
    save_activations_every_epochs = 1  # Set to 1 to save activations every epoch

    # --- 6. File Paths ---
    local_path = "/content/GAN_Outputs_Exp2"
    output_dir = os.path.join(local_path, "outputs")
    checkpoint_dir = os.path.join(local_path, "checkpoints_exp2")
    activation_dir = os.path.join(local_path, "activations_exp2")

    def __init__(self):
        """Creates all necessary output directories to prevent FileNotFoundError."""
        if self.dataset_name == "mnist":
            self.image_channels = 1

        self.samples_dir = os.path.join(self.output_dir, "samples")
        self.monitoring_dir = os.path.join(self.output_dir, "monitoring")
        self.grad_cam_dir = os.path.join(self.output_dir, "grad_cam")

        os.makedirs(self.samples_dir, exist_ok=True)
        os.makedirs(self.monitoring_dir, exist_ok=True)
        os.makedirs(self.grad_cam_dir, exist_ok=True)
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        os.makedirs(self.activation_dir, exist_ok=True)


config = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Outputs will be saved to: {config.local_path}")

Using device: cuda
Outputs will be saved to: /content/GAN_Outputs_Exp2


In [ ]:
# Delete previous run
! rm -rf /content/GAN_Outputs_Final

In [ ]:
# ============================================================================
# DATASET LOADING
# ============================================================================
def get_dataloader():
    transform = transforms.Compose(
        [
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5,) * config.image_channels, (0.5,) * config.image_channels
            ),
        ]
    )
    dataset_class = (
        torchvision.datasets.CIFAR10
        if config.dataset_name == "cifar10"
        else torchvision.datasets.MNIST
    )
    dataset = dataset_class(
        root=config.data_path, train=True, transform=transform, download=True
    )
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
    )


In [ ]:

# ============================================================================
# MODELS (GENERATOR & CRITIC)
# ============================================================================
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.activations = {}
        self.main = nn.Sequential(
            nn.ConvTranspose2d(
                config.latent_dim,
                config.generator_feature_maps * 8,
                4,
                1,
                0,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 8,
                config.generator_feature_maps * 4,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 4,
                config.generator_feature_maps * 2,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 2,
                config.generator_feature_maps,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps,
                config.image_channels,
                3,
                1,
                1,
                bias=False,
            ),
            nn.Tanh(),
        )
        self._initialize_weights()
        self._register_hooks()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.orthogonal_(m.weight, gain=0.8)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

    def _register_hooks(self):
        def get_hook(name):
            def hook(model, input, output):
                self.activations[name] = output.detach()


            return hook

        for i, layer in enumerate(self.main):
            if isinstance(layer, nn.ReLU) or isinstance(layer, nn.Tanh):
                layer.register_forward_hook(get_hook(f"gen_layer_{i}"))

    def forward(self, z):
        return self.main(z)

    def clear_activations(self):
        self.activations.clear()

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.activations = {}
        self.gradients = {}
        self.layer1 = nn.Sequential(
            nn.Conv2d(
                config.image_channels, config.critic_feature_maps, 4, 2, 1, bias=False
            ),
            nn.LeakyReLU(0.2, inplace=False),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(
                config.critic_feature_maps,
                config.critic_feature_maps * 2,
                4,
                2,
                1,
                bias=False,
            ),
            nn.InstanceNorm2d(config.critic_feature_maps * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=False),
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(
                config.critic_feature_maps * 2,
                config.critic_feature_maps * 4,
                4,
                2,
                1,
                bias=False,
            ),
            nn.InstanceNorm2d(config.critic_feature_maps * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=False),
        )
        self.final_conv = nn.Conv2d(
            config.critic_feature_maps * 4, 1, 4, 1, 0, bias=False
        )
        self._initialize_weights()
        self._register_hooks()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.orthogonal_(m.weight, gain=0.8)
            elif isinstance(m, nn.InstanceNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

    def _register_hooks(self):
        def get_activation_hook(name):
            def hook(model, input, output):
                self.activations[name] = output

            return hook

        def get_gradient_hook(name):
            def hook(model, grad_in, grad_out):
                self.gradients[name] = grad_out[0]

            return hook

        self.layer1[0].register_forward_hook(get_activation_hook("layer1"))
        self.layer1[0].register_full_backward_hook(get_gradient_hook("layer1"))
        self.layer2[0].register_forward_hook(get_activation_hook("layer2"))
        self.layer2[0].register_full_backward_hook(get_gradient_hook("layer2"))
        self.layer3[0].register_forward_hook(get_activation_hook("layer3"))
        self.layer3[0].register_full_backward_hook(get_gradient_hook("layer3"))

    def forward(self, x):
        h1 = self.layer1(x)
        h2 = self.layer2(h1)
        h3 = self.layer3(h2)
        out = self.final_conv(h3)
        return out.view(-1)

    def clear_hooks_data(self):
        self.activations.clear()
        self.gradients.clear()

In [ ]:
# ============================================================================
# ANALYSIS TOOLS
# ============================================================================
import os
import numpy as np
import torch
from torchvision.utils import save_image

class ActivationAnalyzer:
    """Collects, analyzes, and saves model activations for offline analysis."""

    def __init__(self, save_dir):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)

    def collect_and_save_epoch_activations(self, epoch, generator, critic, noise_batch, real_batch=None, topk_features=5):
        """
        Saves generator/critic activations and runs channel-wise
        real-vs-fake analysis, probing the top-k feature maps.
        """
        print(f"--- Collecting activations for epoch {epoch} ---")
        generator.eval()
        critic.eval()
        device = next(generator.parameters()).device

        with torch.no_grad():
            generator.clear_activations()
            critic.clear_hooks_data()

            # ---- Forward fake images ----
            fake_imgs = generator(noise_batch.to(device))
            scores_fake = critic(fake_imgs).view(-1)  # per-image critic scores
            fake_acts = {k: v.clone().cpu() for k, v in critic.activations.items()}

            # ---- Forward real images if provided ----
            real_acts = None
            scores_real = None
            if real_batch is not None:
                critic.clear_hooks_data()
                real_imgs = real_batch.to(device)
                scores_real = critic(real_imgs).view(-1)
                real_acts = {k: v.clone().cpu() for k, v in critic.activations.items()}

            # ---- Compute thresholds based on WGAN scores ----
            mean_real = scores_real.mean().item() if scores_real is not None else 0.0
            mean_fake = scores_fake.mean().item()
            w_distance = abs(mean_real - mean_fake)
            margin = w_distance * 0.1

            real_threshold = mean_real - margin
            fake_threshold = mean_fake + margin

            # Select judged real/fake indices
            judged_real_idx = (scores_real > real_threshold).nonzero(as_tuple=True)[0] if scores_real is not None else torch.tensor([], dtype=torch.long)
            judged_fake_idx = (scores_fake < fake_threshold).nonzero(as_tuple=True)[0]

            # ---- Save generator activations ----
            #gen_dir = os.path.join(self.save_dir, f"epoch_{epoch}_generator")
            #os.makedirs(gen_dir, exist_ok=True)
            #for k, v in generator.activations.items():
                #np.save(os.path.join(gen_dir, f"{k}.npy"), v.cpu().numpy())

            # ---- Save critic activations ----
            #crit_dir_fake = os.path.join(self.save_dir, f"epoch_{epoch}_critic_fake")
            #os.makedirs(crit_dir_fake, exist_ok=True)
            #for k, v in fake_acts.items():
                #np.save(os.path.join(crit_dir_fake, f"{k}.npy"), v.numpy())

            # if real_acts is not None:
                # crit_dir_real = os.path.join(self.save_dir, f"epoch_{epoch}_critic_real")
                # os.makedirs(crit_dir_real, exist_ok=True)
                # for k, v in real_acts.items():
                    # np.save(os.path.join(crit_dir_real, f"{k}.npy"), v.numpy())

            # ---- Rank channels by score-based difference ----
            all_acts = {}
            all_scores = {}
            # Combine real and fake acts into one dict with batch dimension
            for k in fake_acts.keys():
                if real_acts is not None and k in real_acts:
                    all_acts[k] = torch.cat([real_acts[k], fake_acts[k]], dim=0)
                else:
                    all_acts[k] = fake_acts[k]
            if real_batch is not None:
                all_scores = torch.cat([scores_real, scores_fake], dim=0)
            else:
                all_scores = scores_fake

            ranked = self.rank_feature_maps_by_scores(all_acts, all_scores, real_threshold, fake_threshold)
            np.save(os.path.join(self.save_dir, f"ranked_features_epoch_{epoch}.npy"), ranked)

            # ---- Probe top-k channels ----
            for feature_name, score in ranked[:topk_features]:
                print(f"Epoch {epoch} | Probing {feature_name} (diff={score:.3f})")
                self.probe_feature_map(epoch, generator, critic, feature_name)

            generator.clear_activations()
            critic.clear_hooks_data()

        torch.cuda.empty_cache()
        print(f"Successfully saved activations for epoch {epoch}")
        generator.train()
        critic.train()

    # ---------- Ranking function ----------
    def rank_feature_maps_by_scores(self, acts, scores, real_threshold, fake_threshold):
        """
        Rank channels by difference between judged-real vs judged-fake mean activations.
        acts: dict[layer] = [N, C, H, W]
        scores: [N] tensor of critic outputs
        """
        judged_real_idx = (scores > real_threshold).nonzero(as_tuple=True)[0]
        judged_fake_idx = (scores < fake_threshold).nonzero(as_tuple=True)[0]

        judged_real_idx = judged_real_idx.cpu()
        judged_fake_idx = judged_fake_idx.cpu()

        diffs = []
        for layer in acts.keys():
            r = acts[layer][judged_real_idx].mean(dim=[0, 2, 3]) if len(judged_real_idx) > 0 else torch.zeros(acts[layer].size(1))
            f = acts[layer][judged_fake_idx].mean(dim=[0, 2, 3]) if len(judged_fake_idx) > 0 else torch.zeros(acts[layer].size(1))
            channel_diffs = torch.abs(r - f).cpu().numpy()
            for ch, val in enumerate(channel_diffs):
                diffs.append((f"{layer}_ch{ch}", float(val)))
        return sorted(diffs, key=lambda x: x[1], reverse=True)

    def probe_feature_map(self, epoch, generator, critic, layer_channel, num_images=1000, topk=16):
        """
        Generate many fake images and find which ones maximally/minimally
        activate a given feature map channel.
        """
        layer, ch = layer_channel.split("_ch")
        ch = int(ch)

        device = next(generator.parameters()).device
        generator.eval()
        critic.eval()

        with torch.no_grad():
            critic.clear_hooks_data()
            z = torch.randn(num_images, generator.main[0].in_channels, 1, 1, device=device)
            imgs = generator(z)
            _ = critic(imgs)  # populates critic.activations
            acts = critic.activations[layer]  # [N, C, H, W]
            scores = acts[:, ch].view(acts.size(0), -1).mean(1)  # per-image mean activation

            top_idx = scores.topk(topk).indices
            low_idx = scores.topk(topk, largest=False).indices

            probe_dir = os.path.join(self.save_dir, f"epoch_{epoch}_probes")
            os.makedirs(probe_dir, exist_ok=True)
            save_image(imgs[top_idx], f"{probe_dir}/top_{layer}_ch{ch}.png", normalize=True, nrow=4)
            save_image(imgs[low_idx], f"{probe_dir}/low_{layer}_ch{ch}.png", normalize=True, nrow=4)

        critic.clear_hooks_data()
        generator.train()
        critic.train()



def run_grad_cam_analysis(generator, critic, num_images=5, epoch_label="final"):
    """
    Generates images and runs Grad-CAM to visualize what the Critic focuses on.
    This is the definitive, corrected version that resolves the KeyError.
    """
    print("\n--- Running Offline Grad-CAM Analysis ---")
    generator.eval()
    critic.eval()

    # Generate a batch of images up front
    noise = torch.randn(num_images, config.latent_dim, 1, 1, device=device)
    images = generator(noise).detach()
    entropies = []  # store per-image entropy values
    for i in range(num_images):
        image_for_cam = images[i:i+1].clone().requires_grad_(True)
        critic.clear_hooks_data()
        score = critic(image_for_cam)
        score.backward()

        gradients = critic.gradients['layer3'][0]
        activations = critic.activations['layer3'][0]
        pooled_gradients = torch.mean(gradients, dim=[1, 2])
        for j in range(activations.shape[0]):
            activations[j, :, :] *= pooled_gradients[j]

        heatmap = torch.mean(activations, dim=0).squeeze()
        heatmap = F.relu(heatmap)
        heatmap /= torch.max(heatmap) if torch.max(heatmap) > 0 else 1.0
        heatmap = heatmap.cpu().detach().numpy()

        # --- Shannon entropy ---
        heatmap_flat = heatmap.flatten()
        if np.var(heatmap_flat) > 1e-6:  # skip nearly flat maps
            hist, _ = np.histogram(heatmap_flat, bins=256, range=(0, 1), density=True)
            hist = hist[hist > 0]
            cam_entropy = entropy(hist, base=2)
            entropies.append(cam_entropy)

        # --- Visualization code ---
        img_np = (image_for_cam[0].permute(1, 2, 0).cpu().detach().numpy() * 0.5) + 0.5
        heatmap_resized = cv2.resize(heatmap, (img_np.shape[1], img_np.shape[0]))
        heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_color_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        superimposed_img = heatmap_color_rgb * 0.4 + img_np * 255 * 0.6
        superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
        ax1.imshow(img_np); ax1.set_title("Generated Image"); ax1.axis('off')
        ax2.imshow(superimposed_img); ax2.set_title("Grad-CAM Overlay"); ax2.axis('off')

        save_path = os.path.join(config.grad_cam_dir, f"grad_cam_epoch_{epoch_label}_img_{i+1}.png")
        plt.savefig(save_path); plt.close(fig)
        # --- Batch-level summary ---
        if entropies:
            avg_entropy = np.mean(entropies)
            print(f"Epoch {epoch_label}: Average Grad-CAM entropy = {avg_entropy:.4f} (N={len(entropies)} valid maps)")
        else:
            print(f"Epoch {epoch_label}: No valid Grad-CAM maps (all flat/null)")
        print(f"Saved Grad-CAM image to {save_path}")

In [ ]:
# ============================================================================
# TRAINING UTILITIES & MONITORING
# ============================================================================
def get_noise_std(epoch):
    if epoch >= config.noise_decay_epochs:
        return config.final_noise_std
    return config.initial_noise_std * (1.0 - (epoch / config.noise_decay_epochs))


def compute_gradient_penalty(C, real, fake):
    alpha = torch.rand(real.size(0), 1, 1, 1, device=device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    c_interp = C(interp)
    grads = torch.autograd.grad(
        outputs=c_interp,
        inputs=interp,
        grad_outputs=torch.ones(c_interp.size(), device=device),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    return ((grads.view(grads.size(0), -1).norm(2, dim=1) - 1) ** 2).mean()


class GANMonitor:
    def __init__(self):
        self.metrics = defaultdict(lambda: deque(maxlen=1000))
        self.health_scores = []

    def log_step(self, s, e, cl, gl, cr, cf):
        self.metrics["step"].append(s)
        self.metrics["epoch"].append(e)
        self.metrics["c_loss"].append(cl)
        self.metrics["g_loss"].append(gl)
        self.metrics["c_real_score"].append(cr.mean().item())
        self.metrics["c_fake_score"].append(cf.mean().item())

    def check_and_plot_health(self, s):
        if len(self.metrics["c_loss"]) < 10:
            return
        c, g = np.mean(list(self.metrics["c_loss"])[-10:]), np.mean(
            list(self.metrics["g_loss"])[-10:]
        )
        h = 100.0
        if c > 0:
            h -= 30
        if abs(g) > 5 * abs(c) and abs(g) > 10:
            h -= 40
        self.health_scores.append({"step": s, "score": max(0, h)})
        fig, ax = plt.subplots(2, 2, figsize=(15, 10))
        steps = list(self.metrics["step"])
        ax[0, 0].plot(steps, list(self.metrics["c_loss"]), label="Critic")
        ax[0, 0].plot(steps, list(self.metrics["g_loss"]), label="Generator")
        ax[0, 0].set_title("Losses")
        ax[0, 0].legend()
        ax[0, 1].plot(steps, list(self.metrics["c_real_score"]), label="Real")
        ax[0, 1].plot(steps, list(self.metrics["c_fake_score"]), label="Fake")
        ax[0, 1].set_title("Average Scores")
        ax[0, 1].legend()
        if self.health_scores:
            hs = [h["step"] for h in self.health_scores]
            hv = [h["score"] for h in self.health_scores]
            ax[1, 0].plot(hs, hv, color="g")
            ax[1, 0].set_title("Training Health")
            ax[1, 0].set_ylim(0, 100)
        fig.delaxes(ax[1, 1])
        plt.suptitle("WGAN Training Monitor")
        plt.tight_layout()
        plt.savefig(f"{config.monitoring_dir}/dashboard_step_{s}.png")
        plt.close()


# ============================================================================
# REVISED REALITY MONITORING METRICS
# ============================================================================
import os, json, csv, torch

class RealityMonitoringMetrics:
    def __init__(self, uncertainty_width_fraction=0.5, margin_fraction=0.1, save_dir=None):
        self.uncertainty_width_fraction = uncertainty_width_fraction
        self.margin_fraction = margin_fraction
        self.save_dir = save_dir
        if self.save_dir is not None:
            os.makedirs(self.save_dir, exist_ok=True)

        # storage for per-batch metrics
        self.epoch_values = {
            "source_confusion": [],
            "false_memory_rate": [],
            "hypercritical_rate": [],
            "d_prime": []
        }

        # store epoch means over training
        self.history = {
            "source_confusion": [],
            "false_memory_rate": [],
            "hypercritical_rate": [],
            "d_prime": []

        }

        self._epoch_counter = 0  # auto-incremented

        # CSV setup if enabled
        if self.save_dir is not None:
            self.csv_path = os.path.join(self.save_dir, "metrics_log.csv")
            if not os.path.exists(self.csv_path):
                with open(self.csv_path, "w", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        "epoch", "source_confusion", "false_memory_rate",
                        "hypercritical_rate", "d_prime"
                    ])

    def compute_error_patterns(self, real_scores, fake_scores):
        with torch.no_grad():
            mean_real = real_scores.mean()
            mean_fake = fake_scores.mean()
            w_distance = mean_real - mean_fake

            std_real = real_scores.std(unbiased=False)
            std_fake = fake_scores.std(unbiased=False)

            # Compute denominator of d'
            denom = torch.sqrt(0.5 * (std_real**2 + std_fake**2))

            if w_distance <= 1e-6:
                metrics = {
                    "source_confusion": 1.0,
                    "false_memory_rate": (fake_scores >= real_scores.mean()).float().mean().item(),
                    "hypercritical_rate": (real_scores <= fake_scores.mean()).float().mean().item()
                }
            else:
                midpoint = (mean_real + mean_fake) / 2.0
                band_half_width = (w_distance * self.uncertainty_width_fraction) / 2.0
                uncertainty_lower = midpoint - band_half_width
                uncertainty_upper = midpoint + band_half_width
                d_prime = (mean_real - mean_fake) / (denom + 1e-12)
                real_uncertain = (real_scores > uncertainty_lower) & (real_scores < uncertainty_upper)
                fake_uncertain = (fake_scores > uncertainty_lower) & (fake_scores < uncertainty_upper)
                source_confusion = ((real_uncertain.float().mean() + fake_uncertain.float().mean()) / 2).item()

                margin = w_distance * self.margin_fraction
                midpoint = (mean_real + mean_fake) / 2.0
                false_memory_threshold = midpoint
                false_memory_rate = (fake_scores > false_memory_threshold).float().mean().item()

                hypercritical_threshold = midpoint
                hypercritical_rate = (real_scores < hypercritical_threshold).float().mean().item()

                metrics = {
                    "source_confusion": source_confusion,
                    "false_memory_rate": false_memory_rate,
                    "hypercritical_rate": hypercritical_rate,
                    "d_prime": d_prime
                }

        for k in metrics:
            self.epoch_values[k].append(metrics[k])
        return metrics

    def finalize_epoch(self):
        epoch_means = {k: float(torch.tensor(v).mean().item() if len(v) > 0 else float('nan'))
                       for k, v in self.epoch_values.items()}

        fa = epoch_means["false_memory_rate"]
        miss = epoch_means["hypercritical_rate"]
        d_prime = epoch_means["d_prime"]

        self.history["source_confusion"].append(epoch_means["source_confusion"])
        self.history["false_memory_rate"].append(fa)
        self.history["hypercritical_rate"].append(miss)
        self.history["d_prime"].append(d_prime)


        record = {
            "epoch": self._epoch_counter + 1,
            "source_confusion": epoch_means["source_confusion"],
            "false_memory_rate": fa,
            "hypercritical_rate": miss,
            "d_prime": d_prime,
        }

        if self.save_dir is not None:
            # JSON
            fname = os.path.join(self.save_dir, f"sdt_epoch_{self._epoch_counter + 1:04d}.json")
            with open(fname, "w") as f:
                json.dump(record, f, indent=2)

            # CSV append
            with open(self.csv_path, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    record["epoch"], record["source_confusion"], record["false_memory_rate"],
                    record["hypercritical_rate"], record["d_prime"]
                ])

        for k in self.epoch_values:
            self.epoch_values[k].clear()

        self._epoch_counter += 1
        return record

    def plot_history(self):
      """Plots the developmental trend of each metric so far."""
      plt.figure(figsize=(8, 5))
      for metric_name, values in self.history.items():
          if values:  # skip empty lists
              plt.plot(range(1, len(values) + 1), values, label=metric_name)
      plt.xlabel("Epoch")
      plt.ylabel("Value")
      plt.title("Reality Monitoring Metrics Over Time")
      plt.legend()
      plt.grid(True)
      plt.show()


In [ ]:
# ============================================================================
# MAIN TRAINING LOOP
# ============================================================================
def trained_generator_vs_untrained_discriminator():
    from google.colab import files
    import os
    import gc

    dataloader = get_dataloader()
    G = Generator().to(device)
    C = Critic().to(device)
    checkpoint_dir = "/content/checkpoints"
    g_path = os.path.join(checkpoint_dir, 'G_epoch_71.pth')
    G.load_state_dict(torch.load(g_path))

    G.train()
    C.train()


    opt_G = torch.optim.Adam(
        G.parameters(),
        lr=config.learning_rate_generator,
        betas=(config.adam_beta1, config.adam_beta2),
    )
    opt_C = torch.optim.Adam(
        C.parameters(),
        lr=config.learning_rate_critic,
        betas=(config.adam_beta1, config.adam_beta2),
    )

    analyzer = ActivationAnalyzer(save_dir=config.activation_dir)
    monitor = GANMonitor()
    rm_metrics = RealityMonitoringMetrics(save_dir="metrics")
    fixed_noise = torch.randn(64, config.latent_dim, 1, 1, device=device)
    global_step = 0

    print("--- Starting WGAN-GP Training ---")
    for epoch in range(config.num_epochs):
        epoch_start = timer()
        for batch_idx, (real_imgs, _) in enumerate(dataloader):
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)
            for _ in range(config.critic_updates_per_generator_update):
                C.zero_grad()
                z = torch.randn(batch_size, config.latent_dim, 1, 1, device=device)
                fake_imgs = G(z).detach()

                c_real = C(real_imgs)
                c_fake = C(fake_imgs)

                if global_step % config.monitor_every_steps == 0:
                  # Compute metrics on noised images
                  with torch.no_grad():

                      errors = rm_metrics.compute_error_patterns(c_real, c_fake)

                  w_distance = c_real.mean().item() - c_fake.mean().item()

                gp = compute_gradient_penalty(C, real_imgs, fake_imgs)
                loss_C = (
                    c_fake.mean() - c_real.mean() + config.gradient_penalty_weight * gp
                )
                loss_C.backward()
                opt_C.step()


            G.zero_grad()
            z = torch.randn(batch_size, config.latent_dim, 1, 1, device=device)
            gen_imgs = G(z)
            c_gen_fake = C(gen_imgs)
            loss_G = -c_gen_fake.mean()
            loss_G.backward()
            opt_G.step()

            if global_step % config.monitor_every_steps == 0:
                log_message = (
                      f"[{epoch+1:02d}/{config.num_epochs}][{global_step:05d}] | "
                      f"Critic Loss: {loss_C.item():+7.3f} | Gen Loss: {loss_G.item():+7.3f} | "
                      f"Scores (Real/Fake): {c_real.mean().item():+6.2f}/{c_fake.mean().item():+6.2f} | "
                      f"RM (Conf/FM): {errors['source_confusion']:.3f}/{errors['false_memory_rate']:.3f} | "
                      f"W-Distance: {w_distance:.2f}"
                  )
                print(log_message)
                monitor.log_step(global_step, epoch, loss_C.item(), loss_G.item(),
                  c_real.detach().cpu(), c_fake.detach().cpu())
                monitor.check_and_plot_health(global_step)




            if global_step % config.save_samples_every_steps == 0:
                with torch.no_grad():
                    G.eval()
                    save_image(
                        G(fixed_noise),
                        f"{config.samples_dir}/s_{global_step}.png",
                        normalize=True,
                        nrow=8,
                    )
                    G.train()
            global_step += 1


        torch.save(G.state_dict(), f"{config.checkpoint_dir}/G_epoch_{epoch+1}_exp1.pth")
        torch.save(C.state_dict(), f"{config.checkpoint_dir}/C_epoch_{epoch+1}_exp1.pth")
        print(f"\nSaved models at epoch {epoch+1}\n")

        if (epoch + 1) % config.save_activations_every_epochs == 0:
            real_batch = next(iter(dataloader))[0]  # one batch of real images
            analyzer.collect_and_save_epoch_activations(epoch+1, G, C, fixed_noise, real_batch, topk_features=5)
        results = rm_metrics.finalize_epoch()
        print(f"Epoch {epoch+1} metrics:", results)
        print(f"Epoch {epoch+1} finished in {timer() - epoch_start:.2f}s.")

        if (epoch + 1) % 25 == 0:
          rm_metrics.plot_history()

        gc.collect()
        torch.cuda.empty_cache()  # if using GPU
    print("--- Training Completed ---")

    print("\nZipping output files for download...")
    output_folder_name = os.path.basename(config.local_path)
    zip_file_name = f"{output_folder_name}.zip"

    # Zip the folder
    !zip -r "{zip_file_name}" "{output_folder_name}"

    print(f"Done! Downloading '{zip_file_name}'...")

    # Automatically download the zip file
    files.download(zip_file_name)

    !zip -r "metrics.zip" "/content/metrics"

    files.download("metrics.zip")

    # print("\nZipping output files for download...")
    # output_folder_name = os.path.basename(config.local_path)
    # !zip -r "{output_folder_name}.zip" "{output_folder_name}"
    # print(f"Done! Find your results in '{output_folder_name}.zip'.")
    # print("You can download it from the Files pane on the left before your session ends.")

In [ ]:
# ============================================================================
# MAIN TRAINING LOOP
# ============================================================================
def trained_discriminator_vs_untrained_generator():
    from google.colab import files
    import os
    import gc

    dataloader = get_dataloader()
    G = Generator().to(device)
    C = Critic().to(device)
    checkpoint_dir = "/content/checkpoints"
    c_path = os.path.join(checkpoint_dir, 'C_epoch_71.pth')
    C.load_state_dict(torch.load(c_path))

    G.train()
    C.train()


    opt_G = torch.optim.Adam(
        G.parameters(),
        lr=config.learning_rate_generator,
        betas=(config.adam_beta1, config.adam_beta2),
    )
    opt_C = torch.optim.Adam(
        C.parameters(),
        lr=config.learning_rate_critic,
        betas=(config.adam_beta1, config.adam_beta2),
    )

    analyzer = ActivationAnalyzer(save_dir=config.activation_dir)
    monitor = GANMonitor()
    rm_metrics = RealityMonitoringMetrics(save_dir="metrics")
    fixed_noise = torch.randn(64, config.latent_dim, 1, 1, device=device)
    global_step = 0

    print("--- Starting WGAN-GP Training ---")
    for epoch in range(config.num_epochs):
        epoch_start = timer()
        for batch_idx, (real_imgs, _) in enumerate(dataloader):
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)
            for _ in range(config.critic_updates_per_generator_update):
                C.zero_grad()
                z = torch.randn(batch_size, config.latent_dim, 1, 1, device=device)
                fake_imgs = G(z).detach()

                c_real = C(real_imgs)
                c_fake = C(fake_imgs)

                if global_step % config.monitor_every_steps == 0:
                  # Compute metrics on noised images
                  with torch.no_grad():

                      errors = rm_metrics.compute_error_patterns(c_real, c_fake)

                  w_distance = c_real.mean().item() - c_fake.mean().item()

                gp = compute_gradient_penalty(C, real_imgs, fake_imgs)
                loss_C = (
                    c_fake.mean() - c_real.mean() + config.gradient_penalty_weight * gp
                )
                loss_C.backward()
                opt_C.step()


            G.zero_grad()
            z = torch.randn(batch_size, config.latent_dim, 1, 1, device=device)
            gen_imgs = G(z)
            c_gen_fake = C(gen_imgs)
            loss_G = -c_gen_fake.mean()
            loss_G.backward()
            opt_G.step()

            if global_step % config.monitor_every_steps == 0:
                log_message = (
                      f"[{epoch+1:02d}/{config.num_epochs}][{global_step:05d}] | "
                      f"Critic Loss: {loss_C.item():+7.3f} | Gen Loss: {loss_G.item():+7.3f} | "
                      f"Scores (Real/Fake): {c_real.mean().item():+6.2f}/{c_fake.mean().item():+6.2f} | "
                      f"RM (Conf/FM): {errors['source_confusion']:.3f}/{errors['false_memory_rate']:.3f} | "
                      f"W-Distance: {w_distance:.2f}"
                  )
                print(log_message)
                monitor.log_step(global_step, epoch, loss_C.item(), loss_G.item(),
                  c_real.detach().cpu(), c_fake.detach().cpu())
                monitor.check_and_plot_health(global_step)




            if global_step % config.save_samples_every_steps == 0:
                with torch.no_grad():
                    G.eval()
                    save_image(
                        G(fixed_noise),
                        f"{config.samples_dir}/s_{global_step}.png",
                        normalize=True,
                        nrow=8,
                    )
                    G.train()
            global_step += 1


        torch.save(G.state_dict(), f"{config.checkpoint_dir}/G_epoch_{epoch+1}_exp2.pth")
        torch.save(C.state_dict(), f"{config.checkpoint_dir}/C_epoch_{epoch+1}_exp2.pth")
        print(f"\nSaved models at epoch {epoch+1}\n")

        if (epoch + 1) % config.save_activations_every_epochs == 0:
            real_batch = next(iter(dataloader))[0]  # one batch of real images
            analyzer.collect_and_save_epoch_activations(epoch+1, G, C, fixed_noise, real_batch, topk_features=5)
        results = rm_metrics.finalize_epoch()
        print(f"Epoch {epoch+1} metrics:", results)
        print(f"Epoch {epoch+1} finished in {timer() - epoch_start:.2f}s.")

        if (epoch + 1) % 25 == 0:
          rm_metrics.plot_history()

        gc.collect()
        torch.cuda.empty_cache()  # if using GPU
    print("--- Training Completed ---")

    print("\nZipping output files for download...")
    output_folder_name = os.path.basename(config.local_path)
    zip_file_name = f"{output_folder_name}.zip"

    # Zip the folder
    !zip -r "{zip_file_name}" "{output_folder_name}"

    print(f"Done! Downloading '{zip_file_name}'...")

    # Automatically download the zip file
    files.download(zip_file_name)

    !zip -r "metrics.zip" "/content/metrics"

    files.download("metrics.zip")

    # print("\nZipping output files for download...")
    # output_folder_name = os.path.basename(config.local_path)
    # !zip -r "{output_folder_name}.zip" "{output_folder_name}"
    # print(f"Done! Find your results in '{output_folder_name}.zip'.")
    # print("You can download it from the Files pane on the left before your session ends.")

In [ ]:
trained_generator_vs_untrained_discriminator()

In [ ]:
trained_discriminator_vs_untrained_generator()

## How to Read the Live Training Logs

When you run the script, a log entry will be printed periodically. This is your primary window into the health and progress of the GAN training. Here's a sample log and a breakdown of what each value means:

**Sample Log:**
`[05/25][03900] | Critic Loss:  -2.510 | Gen Loss: -15.823 | Scores (Real/Fake): +16.20/-15.11 | RM (Conf/FM): 0.045/0.250`

---

### Legend

| Header | What it Measures | Healthy Trend | Red Flag / Sign of Trouble |
| :--- | :--- | :--- | :--- |
| **`[Epoch/Total][Step]`** | Your current position in the training. | N/A | N/A |
| **`Critic Loss`** | The Wasserstein "distance" between real and fake data distributions. | Should be **negative**. Starts near 0, becomes more negative, then trends back toward 0 late in training as the generator improves. | A consistently **positive** value indicates critic failure. Wild oscillations are a sign of instability. |
| **`Gen Loss`** | The generator's ability to fool the critic. It aims to make this as negative as possible. | Should be **negative**. Becomes more negative as the generator improves at fooling the critic. | Stuck near 0 or becomes positive. This means the generator is not learning. |
| **`Scores (Real/Fake)`** | The average raw score the critic gives to real and fake images. The most intuitive metric. | A **widening gap**. The `Real` score should become positive, and the `Fake` score should become negative. Late in training, the `Fake` score should rise back towards the `Real` score. | No gap forms (stuck at 0). The gap collapses or reverses (`Fake` > `Real`). `Real` score is consistently negative. |
| **`RM (Conf/FM)`** | **R**eality **M**onitoring cognitive metrics. | **Conf** (Confusion) may be high at the start/end. **FM** (False Memory) should start at 0 and **rise late in training**. A high FM rate is a sign of a successful generator. | A high **FM** rate early in training indicates critic failure. An **FM** rate stuck at 0 means the generator is never good enough to fool the critic. |

---

## Configuration Guide (`Config` Class)

All tunable parameters for the experiment are located in the `Config` class. Modifying these values is the primary way to tune the model's performance and behavior.

### 1. Dataset and Data Paths

| Variable | Default | Description |
| :--- | :--- | :--- |
| `dataset_name` | `'cifar10'` | Determines which dataset to load. Options: `'cifar10'`, `'mnist'`. |
| `data_path` | `'./data'` | Directory where the script will download and store datasets. |

### 2. Model Architecture

These parameters define the size and complexity of the neural networks.

| Variable | Default | Impact of Changing |
| :--- | :--- | :--- |
| `latent_dim` | `100` | The size of the random noise vector given to the generator. **Increasing** this (e.g., to 128 or 256) can allow for more complex and diverse generated images, but may also make training slightly harder. |
| `generator_feature_maps` | `64` | The base number of filters in the generator's convolutional layers. **Increasing** this (e.g., to 128) increases the generator's "capacity," allowing it to create more detailed images, but at the cost of memory and computation time. |
| `critic_feature_maps` | `64` | The base number of filters in the critic's layers. **Increasing** this makes the critic more powerful and better at spotting fakes, but can make it harder for the generator to keep up. |
| `image_channels` | `3` | The number of color channels in the images (3 for RGB, 1 for grayscale). This is set automatically based on `dataset_name`. |

### 3. WGAN-GP Training Parameters

These control the core optimization process. They are critical for balancing the "game" between the generator and the critic.

| Variable | Default | Impact of Changing |
| :--- | :--- | :--- |
| `batch_size` | `64` | Number of images per training step. **Increasing** this provides more stable gradients but requires more GPU memory. **Decreasing** it can introduce noise into training but saves memory. |
| `num_epochs` | `25` | Total number of passes over the entire dataset. **Increase** this for a full training run (e.g., to 100, 200, or more) to allow the model to fully converge. |
| `learning_rate_generator` | `0.0001` | How quickly the generator adapts. A **higher** value can speed up learning but risks instability. A **lower** value is safer but slower. |
| `learning_rate_critic` | `0.0002` | How quickly the critic adapts. **It's crucial for this to be well-balanced with the generator's LR.** A common strategy (TTUR) is to make it 2x-4x the generator's LR to ensure the critic provides a strong learning signal. |
| `adam_beta1` | `0.0` | Adam optimizer parameter. `0.0` or `0.5` is often recommended for GANs over the default of `0.9` as it can help prevent training oscillations. |
| `critic_updates_per_generator_update` | `1` | The `n_critic` ratio. A value of `1` means a 1:1 update schedule. **Increasing** this (e.g., to 5) gives the critic much more power, which can be useful if the generator is "winning" too easily, but can also cause instability. |

### 4. Stability and Regularization

These are special techniques to keep the training process from failing.

| Variable | Default | Impact of Changing |
| :--- | :--- | :--- |
| `gradient_penalty_weight` | `5` | The `lambda` value for the WGAN-GP loss. This is a crucial regularizer. If training is unstable or the critic loss explodes, **decreasing** this (e.g., to 5 or 2) can help. If the critic isn't learning well, **increasing** it (e.g., to 10) can provide a stronger signal. |
| `gradient_clip_value` | `1.0` | A direct safeguard against exploding gradients. It caps the norm of the gradients. **Decreasing** this (e.g., to 0.5) can help stabilize very volatile training, but setting it too low can hinder learning. `1.0` is a safe, standard value. |
| `initial_noise_std` | `0.2` | The starting amount of noise added to images. This creates a "curriculum" where the initial task is easier. **Increase** this if the model struggles at the very beginning. |
| `noise_decay_epochs` | `15` | How many epochs it takes for the noise to decay to zero. A **longer** decay period gives the model more time on the "easier" curriculum. |

---

In [ ]:
# ============================================================================
# STEP 1: LOAD THE TRAINED GENERATOR
# ============================================================================

# ============================================================================
# STEP 3: DOWNLOAD AND SAVE REAL CIFAR-10 IMAGES
# ============================================================================
!pip install pytorch-fid

import os
import torch
from torchvision.utils import save_image
import shutil
import csv
import numpy as np
from pytorch_fid.fid_score import compute_statistics_of_path, calculate_frechet_distance
from google.colab import files

checkpoint_dir = "/content/GAN_Outputs_Exp2/checkpoints_exp2"
real_images_dir = "/content/real_cifar10_images"

os.makedirs(real_images_dir, exist_ok=True)

print(f"\n--- Saving real CIFAR-10 images to '{real_images_dir}'... ---")
print("(This only needs to be done once per session)")

# Use torchvision to get the dataset
cifar10_real_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
cifar10_loader = DataLoader(cifar10_real_dataset, batch_size=100) # Use a large batch size for speed

saved_count = 0
for i, (images, _) in enumerate(cifar10_loader):
    for j in range(images.size(0)):
        # Important: The FID library expects images in 0-255 range, not normalized.
        # So we don't normalize when saving.
        save_image(images[j], os.path.join(real_images_dir, f"real_img_{saved_count}.png"))
        saved_count += 1
    if (i+1) % 100 == 0:
        print(f"  ... saved {saved_count} / {len(cifar10_real_dataset)} real images")

print(f"Real image saving complete. Saved {saved_count} images.")

# --- Settings ---
EPOCH_INTERVAL = 1    # Run FID every 5 epochs
START_EPOCH = 1
END_EPOCH = 90
num_eval_images = 10000  # Standard for FID
fake_images_dir = "/content/generated_images_for_fid"
real_stats_file = "real_stats_cifar10.npz"
os.makedirs(fake_images_dir, exist_ok=True)

from pytorch_fid.inception import InceptionV3

# Create the InceptionV3 model with default dims=2048
block_idx = InceptionV3.BLOCK_INDEX_BY_DIM[2048]
inception_model = InceptionV3([block_idx]).to(device)

# --- Step 1: Precompute or load real stats ---
if os.path.exists(real_stats_file):
    print(f"Loading cached real stats from {real_stats_file}")
    real_stats = np.load(real_stats_file)
    real_mu, real_sigma = real_stats["mu"], real_stats["sigma"]
else:
    print(f"Computing real stats for {real_images_dir} (this will take a few minutes)...")
    real_mu, real_sigma = compute_statistics_of_path(
        real_images_dir,
        model=inception_model,
        batch_size=128,
        device=device,
        dims=2048
    )
    np.savez(real_stats_file, mu=real_mu, sigma=real_sigma)
    print(f"Saved real stats to {real_stats_file}")

# --- Step 2: Iterate over epochs ---
fid_results = {}

for epoch in range(START_EPOCH, END_EPOCH + 1, EPOCH_INTERVAL):
    print(f"\n==============================")
    print(f"Epoch {epoch}: Loading Generator")
    print(f"==============================")

    # --- Load Generator ---
    try:
        G_eval = Generator().to(device)
        checkpoint_path = os.path.join(checkpoint_dir, f"G_epoch_{epoch}_exp1.pth")
        G_eval.load_state_dict(torch.load(checkpoint_path, map_location=device))
        G_eval.eval()
        print("Generator model loaded successfully.")
    except FileNotFoundError:
        print(f"WARNING: Could not load checkpoint for epoch {epoch}. Skipping.")
        continue

    # --- Generate Fake Images ---
    print(f"\n--- Generating {num_eval_images} fake images for epoch {epoch} ---")
    os.makedirs(fake_images_dir, exist_ok=True)

    with torch.no_grad():
        for i in range(0, num_eval_images, config.batch_size):
            noise = torch.randn(config.batch_size, config.latent_dim, 1, 1, device=device)
            fake_imgs = G_eval(noise)
            for j in range(fake_imgs.size(0)):
                current_idx = i + j
                if current_idx >= num_eval_images:
                    break
                save_path = os.path.join(fake_images_dir, f"img_{current_idx}.png")
                save_image(fake_imgs[j], save_path, normalize=True)
            if (i + config.batch_size) % 1000 < config.batch_size:
                print(f"  ... generated {i + config.batch_size} / {num_eval_images} images")

    print("Fake image generation complete.")

    # --- Step 3: Compute FID (only fake stats) ---
    print("\n--- Calculating FID score... ---")
    fake_mu, fake_sigma = compute_statistics_of_path(
        fake_images_dir,
        model=inception_model,
        batch_size=128,
        device=device,
        dims=2048
    )

    fid_value = calculate_frechet_distance(real_mu, real_sigma, fake_mu, fake_sigma)
    fid_results[epoch] = fid_value
    print(f"FID for epoch {epoch}: {fid_value:.4f}")

    # --- Step 4: Clean up generated images ---
    shutil.rmtree(fake_images_dir)

# --- Step 5: Save results to CSV ---
csv_path = "fid_results_untrainedDisc.csv"
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Epoch", "FID"])
    for ep, fid in fid_results.items():
        writer.writerow([ep, fid])

print(f"\nFID results saved to {csv_path}")

files.download(csv_path)

# --- Print summary ---
print("\n==============================")
print("FID Results by Epoch")
print("==============================")
for ep, fid in fid_results.items():
    print(f"Epoch {ep}: FID = {fid:.4f}")




--- Saving real CIFAR-10 images to '/content/real_cifar10_images'... ---
(This only needs to be done once per session)
  ... saved 10000 / 50000 real images
  ... saved 20000 / 50000 real images
  ... saved 30000 / 50000 real images
  ... saved 40000 / 50000 real images
  ... saved 50000 / 50000 real images
Real image saving complete. Saved 50000 images.
Loading cached real stats from real_stats_cifar10.npz

Epoch 1: Loading Generator
Generator model loaded successfully.

--- Generating 10000 fake images for epoch 1 ---
  ... generated 1024 / 10000 images
  ... generated 2048 / 10000 images
  ... generated 3008 / 10000 images
  ... generated 4032 / 10000 images
  ... generated 5056 / 10000 images
  ... generated 6016 / 10000 images
  ... generated 7040 / 10000 images
  ... generated 8000 / 10000 images
  ... generated 9024 / 10000 images
  ... generated 10048 / 10000 images
Fake image generation complete.

--- Calculating FID score... ---


100%|██████████| 79/79 [00:16<00:00,  4.81it/s]


FID for epoch 1: 56.9856

Epoch 2: Loading Generator
Generator model loaded successfully.

--- Generating 10000 fake images for epoch 2 ---
  ... generated 1024 / 10000 images
  ... generated 2048 / 10000 images
  ... generated 3008 / 10000 images
  ... generated 4032 / 10000 images
  ... generated 5056 / 10000 images
  ... generated 6016 / 10000 images
  ... generated 7040 / 10000 images
  ... generated 8000 / 10000 images
  ... generated 9024 / 10000 images
  ... generated 10048 / 10000 images
Fake image generation complete.

--- Calculating FID score... ---


100%|██████████| 79/79 [00:16<00:00,  4.78it/s]


FID for epoch 2: 64.3601

Epoch 3: Loading Generator
Generator model loaded successfully.

--- Generating 10000 fake images for epoch 3 ---
  ... generated 1024 / 10000 images
  ... generated 2048 / 10000 images
  ... generated 3008 / 10000 images
  ... generated 4032 / 10000 images
  ... generated 5056 / 10000 images
  ... generated 6016 / 10000 images
  ... generated 7040 / 10000 images
  ... generated 8000 / 10000 images
  ... generated 9024 / 10000 images
  ... generated 10048 / 10000 images
Fake image generation complete.

--- Calculating FID score... ---


...

100%|██████████| 79/79 [00:16<00:00,  4.80it/s]


FID for epoch 90: 42.9952

FID results saved to fid_results_untrainedDisc.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


FID Results by Epoch
Epoch 1: FID = 56.9856
Epoch 2: FID = 64.3601
Epoch 3: FID = 69.5814
Epoch 4: FID = 64.4804
Epoch 5: FID = 62.2977
Epoch 6: FID = 68.0107
Epoch 7: FID = 74.6720
Epoch 8: FID = 72.2203
Epoch 9: FID = 64.4652
Epoch 10: FID = 68.2734
Epoch 11: FID = 64.1966
Epoch 12: FID = 58.0019
Epoch 13: FID = 54.0494
Epoch 14: FID = 62.4074
Epoch 15: FID = 58.1930
Epoch 16: FID = 54.9967
Epoch 17: FID = 55.4654
Epoch 18: FID = 52.5867
Epoch 19: FID = 54.5298
Epoch 20: FID = 50.5486
Epoch 21: FID = 48.5616
Epoch 22: FID = 49.6496
Epoch 23: FID = 53.2903
Epoch 24: FID = 53.1522
Epoch 25: FID = 46.9207
Epoch 26: FID = 49.2369
Epoch 27: FID = 44.7307
Epoch 28: FID = 47.1880
Epoch 29: FID = 44.2041
Epoch 30: FID = 47.3711
Epoch 31: FID = 44.6727
Epoch 32: FID = 46.6782
Epoch 33: FID = 47.0295
Epoch 34: FID = 47.6042
Epoch 35: FID = 45.2340
Epoch 36: FID = 43.9713
Epoch 37: FID = 46.1805
Epoch 38: FID = 49.8449
Epoch 39: FID = 45.6117
Epoch 40: FID = 44.8558
Epoch 41: FID = 45.7344
Epo